In [19]:
# WhatsApp Auto-Reply Bot – Automated Reply Engine

## NitroXhift Studios Solutions

#Individual Week Contribution

#This notebook demonstrates the AI/NLP matching logic used to map inbound user messages to the most relevant FAQ response.

In [5]:
import pandas as pd
import numpy as np
import re

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity


In [7]:
df = pd.read_csv("../data/faq_dataset.csv")

df

,category,keywords,response
0,Working Hours,"hours,working hours,open,opening time,office h...","Our working hours are Monday to Friday, 9 AM t..."
1,Contact Information,"contact,phone,email,reach,contact information",You can contact our support team through the p...
2,Services,"services,service,offer,provide,offerings","We provide AI/ML, Web Development, Mobile App ..."
3,Pricing,"price,pricing,cost,fee,charges,how much",Our pricing depends on the service and project...
4,Registration,"register,registration,sign up,signup,create ac...",You can register by providing the required inf...
5,Internship,"internship,intern,training,career,job opportunity",We offer internship opportunities for students...
6,Support,"support,help,problem,issue,technical problem",Our support team can assist you with technical...
7,Location,"location,located,where,office,address",Our team operates online and serves clients re...
8,Payment Methods,"payment,pay,credit card,bank transfer,payment ...",Available payment methods depend on the servic...
9,Cancellation Policy,"cancel,cancellation,refund,cancel service","For cancellation and refund information, pleas..."


In [9]:
df["search_text"] = (
    df["category"].fillna("") + " " +
    df["keywords"].fillna("")
)

df[["category", "search_text"]]

,category,search_text
0,Working Hours,"Working Hours hours,working hours,open,opening..."
1,Contact Information,"Contact Information contact,phone,email,reach,..."
2,Services,"Services services,service,offer,provide,offerings"
3,Pricing,"Pricing price,pricing,cost,fee,charges,how much"
4,Registration,"Registration register,registration,sign up,sig..."
5,Internship,"Internship internship,intern,training,career,j..."
6,Support,"Support support,help,problem,issue,technical p..."
7,Location,"Location location,located,where,office,address"
8,Payment Methods,"Payment Methods payment,pay,credit card,bank t..."
9,Cancellation Policy,"Cancellation Policy cancel,cancellation,refund..."


In [11]:
vectorizer = TfidfVectorizer(
    lowercase=True,
    stop_words="english"
)

faq_vectors = vectorizer.fit_transform(
    df["search_text"]
)

print("FAQ Vector Shape:", faq_vectors.shape)

FAQ Vector Shape: (10, 52)


In [13]:
def get_reply(message):

    cleaned_message = message.lower()

    cleaned_message = re.sub(
        r"[^\w\s]",
        "",
        cleaned_message
    )

    message_vector = vectorizer.transform(
        [cleaned_message]
    )

    similarities = cosine_similarity(
        message_vector,
        faq_vectors
    )

    best_index = similarities.argmax()

    best_score = similarities[0][best_index]

    threshold = 0.15

    if best_score >= threshold:

        return {
            "Category":
                df.iloc[best_index]["category"],

            "Response":
                df.iloc[best_index]["response"],

            "Confidence":
                round(float(best_score), 2)
        }

    return {
        "Category": "Unknown",

        "Response":
            "Sorry, I couldn't find an answer to your question. "
            "Please contact our support team for further assistance.",

        "Confidence":
            round(float(best_score), 2)
    }

In [15]:
get_reply("What are your working hours?")

{'Category': 'Working Hours',
 'Response': 'Our working hours are Monday to Friday, 9 AM to 5 PM.',
 'Confidence': 0.82}

In [17]:
get_reply("How much does your service cost?")

{'Category': 'Pricing',
 'Response': 'Our pricing depends on the service and project requirements. Please contact our support team for a detailed quotation.',
 'Confidence': 0.27}

In [21]:
get_reply("Where is your office located?")

{'Category': 'Location',
 'Response': 'Our team operates online and serves clients remotely.',
 'Confidence': 0.51}

In [23]:
get_reply("Who won the football match yesterday?")


{'Category': 'Unknown',
 'Response': "Sorry, I couldn't find an answer to your question. Please contact our support team for further assistance.",
 'Confidence': 0.0}

In [25]:
get_reply("Where is your office located?")


{'Category': 'Location',
 'Response': 'Our team operates online and serves clients remotely.',
 'Confidence': 0.51}